In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import timedelta
import os
from nielsen_shared import (
    BASE_DIR,
    get_brand_form,
    get_scanner_dir,
    get_panel_dir,
    get_hierarchy_upcs,
    get_table,
    transform_brand,
    get_week_end
)
from config_plt import save_my_fig

pd.set_option("display.max_rows", 1000)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 500)

# Create Basic Panel

In [2]:
def process_panel_data(year):
    print(f"Processing panel data for {year}")
    panel_dir = get_panel_dir(year)
    
    # Brand-form combinations
    df_brand_form = get_brand_form()
    brand_forms = df_brand_form["brand_form"].tolist()
    
    # UPC attributes
    df_hierarchy, upcs = get_hierarchy_upcs(year)
    df_hierarchy = df_hierarchy.rename({"sub_category": "form"}, axis=1)
    df_attributes = get_table("productattributes.tsv", year, upcs)
    df_attributes["brand"] = transform_brand(df_attributes["BRAND"])
    df_attributes_expanded = df_attributes.merge(df_hierarchy, on=["upc", "year"], how="left")
    df_attributes_expanded["brand_form"] = df_attributes_expanded["brand"] + "-" + df_attributes_expanded["form"]
    df_attributes_expanded["brand_form"] = np.where(df_attributes_expanded["brand_form"].isin(brand_forms), df_attributes_expanded["brand_form"], "OTHER")
    df_attributes_expanded.to_csv(os.path.join(panel_dir, "panel_attributes.csv"), index=False)
    
    # Purchases
    df_purchase = get_table("purchase.tsv", year, upcs, panel=True)  
    df_purchase_expanded = (df_purchase
        .rename({"panel_year": "year"}, axis=1)
        .merge(df_attributes_expanded, on=["upc", "year"], how="inner"))
    
    # Sum to brand-form level
    df_purchase_expanded["total_net_price"] = df_purchase_expanded["total_price_paid"] - df_purchase_expanded["coupon_value"]
    df_purchase_bf = df_purchase_expanded.groupby(["year", "trip_code_uc", "brand_form"])[["total_price_paid", "total_net_price", "coupon_value"]].sum().reset_index()
    df_purchase_bf["u"] = np.random.default_rng(seed=42).random(df_purchase_bf.shape[0])   
    df_purchase_bf = df_purchase_bf.sort_values(by=["year", "trip_code_uc", "total_net_price", "u"], ascending=False)
    df_purchase_bf["bf_priority"] = df_purchase_bf.groupby(["year", "trip_code_uc"]).cumcount()
    df_choice_bf = df_purchase_bf[df_purchase_bf["bf_priority"] == 0]

    # Also sum to brand and form levels individually for creating loyalty variables
    df_purchase_brand = df_purchase_expanded.groupby(["year", "trip_code_uc", "brand"])["total_net_price"].sum().reset_index()
    df_purchase_brand["u"] = np.random.default_rng(seed=42).random(df_purchase_brand.shape[0])   
    df_purchase_brand = df_purchase_brand.sort_values(by=["year", "trip_code_uc", "total_net_price", "u"], ascending=False)
    df_purchase_brand["brand_priority"] = df_purchase_brand.groupby(["year", "trip_code_uc"]).cumcount()
    df_choice_brand = df_purchase_brand[df_purchase_brand["brand_priority"] == 0]

    df_purchase_form = df_purchase_expanded.groupby(["year", "trip_code_uc", "form"])["total_net_price"].sum().reset_index()
    df_purchase_form["u"] = np.random.default_rng(seed=42).random(df_purchase_form.shape[0])   
    df_purchase_form = df_purchase_form.sort_values(by=["year", "trip_code_uc", "total_net_price", "u"], ascending=False)
    df_purchase_form["form_priority"] = df_purchase_form.groupby(["year", "trip_code_uc"]).cumcount()
    df_choice_form = df_purchase_form[df_purchase_form["form_priority"] == 0]

    df_choice = (df_choice_bf
        .merge(df_choice_brand[["year", "trip_code_uc", "brand"]], on=["year", "trip_code_uc"], how="left")
        .merge(df_choice_form[["year", "trip_code_uc", "form"]], on=["year", "trip_code_uc"], how="left"))
    
    # Compute percent of trips with multiple brand_forms
    percent_trips_repeat = (df_purchase_bf.groupby("trip_code_uc")["bf_priority"].max() > 0).mean()
    print(f"Of all trips with at least 1 laundry purchase, {100*percent_trips_repeat:.2f}% have 2+ brand_forms")
        
    # Join in additional trip information
    df_trip = get_table("trip.tsv", year, panel=True).rename({"panel_year": "year"}, axis=1)
    df = df_trip.merge(df_choice, on=["year", "trip_code_uc"], how="inner")
    
    # Final processing
    df = df[("2021-01-03" <= df["purchase_date"]) & (df["purchase_date"] < "2024-12-29")]  # Take only full weeks, starting on Sundays
    df["week_end"] = get_week_end(df["purchase_date"])
    selected_columns = ["household_code", "trip_code_uc", "purchase_date", "week_end", "year", "store_code_uc", "brand_form", "brand", "form", "total_price_paid", "total_net_price", "coupon_value"]
    df = df[selected_columns]
    
    # Save final dataframe
    df.to_csv(os.path.join(panel_dir, "panel_purchases.csv"), index=False)

    return df

In [3]:
years = ["2021", "2022", "2023", "2024"]
for year in years:
    process_panel_data(year)

Processing panel data for 2021
Of all trips with at least 1 laundry purchase, 5.18% have 2+ brand_forms
Processing panel data for 2022
Of all trips with at least 1 laundry purchase, 4.53% have 2+ brand_forms
Processing panel data for 2023
Of all trips with at least 1 laundry purchase, 4.77% have 2+ brand_forms
Processing panel data for 2024
Of all trips with at least 1 laundry purchase, 4.61% have 2+ brand_forms


In [4]:
df_panel = None
for year in years:
    print(year)
    panel_dir = get_panel_dir(year)
    df_year = pd.read_csv(os.path.join(panel_dir, "panel_purchases.csv"))
    if df_panel is None:
        df_panel = df_year
    else:
        df_panel = pd.concat([df_panel, df_year], axis=0)
        del df_year

df_panel["trip_id"] = df_panel["year"].astype(str) + "-" + df_panel["trip_code_uc"].astype(str)
print("Saving combined data")
df_panel.to_csv(os.path.join(BASE_DIR, "combined_panel_purchases.csv"), index=False)
del df_panel

2021
2022
2023
2024
Saving combined data


# Create Price/Promotion Variables

In [5]:
# Convert units to loads and get brand-forms by upc-year
def get_attributes(year):
    panel_dir = get_panel_dir(year)
    df = pd.read_csv(os.path.join(panel_dir, "panel_attributes.csv"), low_memory=False)
    df = df[["upc", "year", "brand_form", "BASE_SIZE"]]
    return df

years = ["2021", "2022", "2023", "2024"]
df_attributes_list = [get_attributes(year) for year in years]
df_attributes = pd.concat(df_attributes_list, axis=0)
base_size_split = df_attributes["BASE_SIZE"].str.split()
df_attributes["size_value"] = base_size_split.apply(lambda x: float(x[0]))
df_attributes["size_units"] = base_size_split.apply(lambda x: " ".join(x[1:]))
df_attributes["loads"] = df_attributes["size_value"] /  np.where(df_attributes["size_units"] == "OUNCE", 1.1, 1.)
df_attributes["brand_form"] = df_attributes["brand_form"].astype("category")
df_attributes = df_attributes[["upc", "year", "brand_form", "loads"]]

In [6]:
# Load other data sets
df_panel = pd.read_csv(os.path.join(BASE_DIR, "combined_panel_purchases.csv"))
df_panel["week_end"] = pd.to_datetime(df_panel["week_end"])
df_panel["purchase_date"] = pd.to_datetime(df_panel["purchase_date"])
df_store_week = df_panel[["store_code_uc", "week_end"]].drop_duplicates().reset_index(drop=True)
df_week = df_store_week[["week_end"]].drop_duplicates().reset_index(drop=True)
store_codes = df_store_week["store_code_uc"].unique()
df_brand_form = get_brand_form()
brand_forms = df_brand_form["brand_form"].tolist()
brands = [b for b in df_brand_form["brand"].unique() if b != "OTHER"]
forms = df_brand_form[~df_brand_form["form"].isna()]["form"].unique().tolist()

In [7]:
# Create prices one year at a time, summing over UPCs
def compute_prices(year):
    print(f"Computing prices for year {year}")
    df_hierarchy, upcs = get_hierarchy_upcs(year)
    df_laundry = get_table("LAUNDRY CARE.tsv", year, upcs, panel=False)
    df_laundry["year"] = int(year)
    df_laundry["week_end"] = pd.to_datetime(df_laundry["week_end"])
    df_laundry["promotion"] = ((df_laundry["feature"] + df_laundry["display"]) > 0).astype(int)
    df_laundry = df_laundry[["store_code_uc", "upc", "year", "week_end", "units", "price", "promotion"]]
    df_laundry_filtered = df_laundry[df_laundry["store_code_uc"].isin(store_codes)]  # Filter out stores that don't appear in panel
    df_merged = df_laundry_filtered.merge(df_attributes, on=["upc", "year"], how="inner")
    df_merged["total_loads"] = df_merged["units"] * df_merged["loads"]
    df_merged["total_price"] = df_merged["units"] * df_merged["price"]
    df_load_sums = (df_merged
        .groupby(["store_code_uc", "brand_form", "week_end"], observed=True)["total_loads"]
        .sum()
        .reset_index()
        .rename({"total_loads": "total_loads_sum"}, axis=1))
    df_load_sums = df_load_sums[df_load_sums["total_loads_sum"] > 0]
    df_weights = df_merged.merge(df_load_sums, on=["store_code_uc", "brand_form", "week_end"], how="inner")
    df_weights["weight"] = df_weights["total_loads"] / df_weights["total_loads_sum"]
    df_weights["weighted_price_per_load"] = df_weights["weight"] * df_weights["price"] / df_weights["loads"]
    df_weights["weighted_promotion"] = df_weights["weight"] * df_weights["promotion"]
    df_prices = df_weights.groupby(["store_code_uc", "brand_form", "week_end"], observed=True)[["weighted_price_per_load", "weighted_promotion"]].sum().reset_index()
    scanner_dir = get_scanner_dir(year)
    df_prices.to_csv(os.path.join(scanner_dir, "prices.csv"), index=False)
    return df_prices

In [8]:
for year in years:
    compute_prices(year)

Computing prices for year 2021
Computing prices for year 2022
Computing prices for year 2023
Computing prices for year 2024


In [9]:
df_prices = None
for year in years:
    print(year)
    scanner_dir = get_scanner_dir(year)
    df_year = pd.read_csv(os.path.join(scanner_dir, "prices.csv"))
    if df_prices is None:
        df_prices = df_year
    else:
        df_prices = pd.concat([df_prices, df_year], axis=0)
        del df_year
df_prices["week_end"] = pd.to_datetime(df_prices["week_end"])

2021
2022
2023
2024


In [10]:
print("Saving combined data")
df_prices.to_csv(os.path.join(BASE_DIR, "combined_prices.csv"), index=False)

Saving combined data


# Add Coupon Variables

In [11]:
# Create coupon variables
df_week_bf = df_panel.groupby(["week_end", "brand_form"])[["total_price_paid", "coupon_value"]].sum().reset_index()
df_week_bf_scaffold = df_week.merge(df_brand_form, how="cross")
df_week_bf = df_week_bf_scaffold.merge(df_week_bf, on=["week_end", "brand_form"], how="left").fillna(0).sort_values(by=["week_end", "brand_form"]).reset_index(drop=False)
df_week_bf["coupon_proportion"] = df_week_bf["coupon_value"] / df_week_bf["total_price_paid"]

In [12]:
# Plot coupon variables
fig, ax = plt.subplots(4, 3, figsize=(12, 12))
for i, bf in enumerate(brand_forms):
    ax_i = ax.item(i)
    # ax_i.set_title(bf)
    df_week_bf_filtered = df_week_bf[df_week_bf["brand_form"] == bf]
    ax_i.plot(df_week_bf_filtered["week_end"], df_week_bf_filtered["coupon_proportion"])
    ax_i.tick_params(axis='x', labelrotation=30)
fig.tight_layout()
save_my_fig(fig, "coupon_proportions")
plt.close()

./figures/coupon_proportions


In [13]:
rho_coupon = 0.5
df_week_bf_smooth = None
for bf in brand_forms:
    is_bf = df_week_bf["brand_form"] == bf
    df_week_bf_filtered = df_week_bf[is_bf].copy()
    smoothed_coupon_proportions = np.zeros(df_week_bf_filtered.shape[0])
    last_value = 0.
    for i, row in enumerate(df_week_bf_filtered.itertuples()):
        this_value = rho_coupon*last_value + (1.-rho_coupon)*row.coupon_proportion
        smoothed_coupon_proportions[i] = this_value
        last_value = this_value

    df_week_bf_filtered["smoothed_coupon_proportion"] = smoothed_coupon_proportions
    if df_week_bf_smooth is None:
        df_week_bf_smooth = df_week_bf_filtered
    else:
        df_week_bf_smooth = pd.concat([df_week_bf_smooth, df_week_bf_filtered], axis=0)

In [14]:
# Plot coupon variables
fig, ax = plt.subplots(4, 3, figsize=(12, 12))
for i, bf in enumerate(brand_forms):
    ax_i = ax.item(i)
    # ax_i.set_title(bf)
    df_week_bf_filtered = df_week_bf_smooth[df_week_bf_smooth["brand_form"] == bf]
    ax_i.plot(df_week_bf_filtered["week_end"], df_week_bf_filtered["smoothed_coupon_proportion"])
    ax_i.tick_params(axis='x', labelrotation=30)
fig.tight_layout()
save_my_fig(fig, "smoothed_coupon_proportions")
plt.close()

./figures/smoothed_coupon_proportions


In [15]:
df_smoothed_coupon_proportions = df_week_bf_smooth.pivot(index=["week_end"], columns=["brand_form"], values="smoothed_coupon_proportion")
df_smoothed_coupon_proportions.columns = [f"coupon: {c}" for c in df_smoothed_coupon_proportions.columns]
df_smoothed_coupon_proportions = df_smoothed_coupon_proportions.reset_index()

# Merge Panel with Prices

In [16]:
# Fill in missing prices
df_store_week_bf = df_store_week.merge(df_brand_form, how="cross")
for week_delta in [-2, -1, 0, 1, 2]:
    print(week_delta)
    df_prices_delta = df_prices.copy().rename({
        "weighted_price_per_load": f"weighted_price_per_load_{-week_delta}",
        "weighted_promotion": f"weighted_promotion_{-week_delta}"
    }, axis=1)
    df_prices_delta["week_end"] = df_prices_delta["week_end"] + timedelta(days=7*week_delta)
    df_store_week_bf = df_store_week_bf.merge(df_prices_delta, on=["store_code_uc", "brand_form", "week_end"], how="left")

-2
-1
0
1
2


In [17]:
for c in ["weighted_price_per_load", "weighted_promotion"]:
    df_store_week_bf[c] = (df_store_week_bf[f"{c}_0"]
        .fillna(df_store_week_bf[f"{c}_-1"])
        .fillna(df_store_week_bf[f"{c}_1"])
        .fillna(df_store_week_bf[f"{c}_-2"])
        .fillna(df_store_week_bf[f"{c}_2"]))
df_prices_filled = df_store_week_bf[["store_code_uc", "week_end", "brand_form", "weighted_price_per_load", "weighted_promotion"]]
df_prices_filled = df_prices_filled[(~df_prices_filled["weighted_price_per_load"].isna()) & (~df_prices_filled["weighted_promotion"].isna())]

In [18]:
# Pivot prices into wide (column) format
df_prices_pivoted = df_prices_filled.pivot(index=["store_code_uc", "week_end"], columns=["brand_form"]).reset_index()
df_prices_pivoted_store_week = df_prices_pivoted[["store_code_uc", "week_end"]]
df_prices_pivoted_store_week.columns = ["store_code_uc", "week_end"]
df_prices_pivoted_prices = df_prices_pivoted.loc[:, "weighted_price_per_load"]
price_columns = [f"price: {c}" for c in df_prices_pivoted_prices.columns]
df_prices_pivoted_prices.columns = price_columns
df_prices_pivoted_promotion = df_prices_pivoted.loc[:, "weighted_promotion"]
promotion_columns = [f"promotion: {c}" for c in df_prices_pivoted_promotion.columns]
df_prices_pivoted_promotion.columns = promotion_columns
df_prices_pivoted = pd.concat([
        df_prices_pivoted_store_week,
        df_prices_pivoted_prices,
        df_prices_pivoted_promotion
    ], axis=1)

In [19]:
# Merge purchases with prices
df_panel_prices = df_panel.merge(df_prices_pivoted, on=["store_code_uc", "week_end"], how="inner").reset_index(drop=True).merge(df_smoothed_coupon_proportions, on="week_end")
has_tide_liquid = ~df_panel_prices["price: TIDE-LIQUID"].isna()
df_panel_prices = df_panel_prices[has_tide_liquid].reset_index(drop=True)

In [20]:
# Visualize available prices
available_bfs = (~df_panel_prices[price_columns].isna()).sum(axis=1)
available_bfs_counts = available_bfs.value_counts().sort_index()
fig, ax = plt.subplots()
ax.barh(available_bfs_counts.index, available_bfs_counts)
for i in available_bfs_counts.index:
    ax.text(available_bfs_counts[i]+1000, i-0.25, f"{available_bfs_counts[i]:,}")
ax.set_yticks(np.arange(19), np.arange(19))
ax.set_ylabel("# Available Prices (max=18)")
ax.set_xlabel("# Purchases")
ax.set_xlim(0, available_bfs_counts.max()*1.15)
ax.set_title("Histogram of Available Prices")
save_my_fig(fig, "available_prices")
plt.close()

./figures/available_prices


In [21]:
percent_available = (~df_panel_prices[price_columns].isna()).mean(axis=0).sort_values(ascending=True)
# percent_available.index = [c[7:] for c in percent_available.index]
percent_available.index = [f"Brand {i}" for i in range(len(percent_available))]

fig, ax = plt.subplots()
ax.barh(percent_available.index, percent_available)
for i in percent_available.index:
    ax.text(percent_available[i]+0.02, i, f"{percent_available[i]*100:.1f}%")
ax.set_xlabel("% Available")
ax.set_ylabel("BRAND-FORM")
ax.set_xlim(0, 1.2)
ax.set_title("Histogram of Available Prices")
save_my_fig(fig, "percent_available")
plt.close()

./figures/percent_available


In [22]:
price_excluded = np.zeros(df_panel_prices.shape[0], dtype=bool)
for i, row in df_panel_prices.iterrows():
    bf = row["brand_form"]
    price_excluded[i] = np.isnan(row[f"price: {bf}"])
num_missing_price = price_excluded.sum()
print(num_missing_price)
df_panel_prices["u"] = np.random.default_rng(seed=42).random(df_panel_prices.shape[0])
df_panel_prices_filtered = df_panel_prices[(~price_excluded) & (available_bfs >= 2)].sort_values(by=["household_code", "purchase_date", "u"]).reset_index(drop=True).copy()

307


# Add Loyalty Variables

In [23]:
def create_loyalty_df(panel_df, column, values, prefix, rho=0.9):
    k = len(values)
    lookup = {}
    for i in range(k):
        a = np.zeros(k)
        a[i] = 1.
        lookup[values[i]] = a.copy()
    
    loyalty = np.zeros((panel_df.shape[0], k))
    lag_loyalty = loyalty.copy()
    last_household_code = -1
    for i, row in enumerate(panel_df.itertuples()):
        household_code = row.household_code
        is_new_household = household_code != last_household_code
        if is_new_household:
            last_loyalty = np.zeros(k)
        else:
            last_loyalty = loyalty[i-1]
        lag_loyalty[i] = last_loyalty.copy()
        column_value = getattr(row, column)
        curr_value = lookup.get(column_value, np.zeros(k))
        loyalty[i] = (1.-rho)*curr_value + rho*last_loyalty
        last_household_code = household_code
    
    df_loyalty = pd.DataFrame(data=lag_loyalty, columns=[f"{prefix}: {v}" for v in values])
    return df_loyalty

df_loyalty_bf = create_loyalty_df(df_panel_prices_filtered, "brand_form", brand_forms, "loyalty")
df_loyalty_brand = create_loyalty_df(df_panel_prices_filtered, "brand", brands, "brand")
df_loyalty_form = create_loyalty_df(df_panel_prices_filtered, "form", forms, "form")
df_panel_loyalty = pd.concat([df_panel_prices_filtered, df_loyalty_bf, df_loyalty_brand, df_loyalty_form], axis=1)

# Save Data

In [24]:
df_panel_loyalty.to_csv(os.path.join(BASE_DIR, "full_panel.csv"), index=False)